#### Anomaly detection 
- amount
- category
- user_average_amount
- category_average_amount
- transaction_frequency
- balance
- day_of_week
- transaction_hour
- is_weekend
- amount_vs_user_avg
- amount_vs_category_avg
- recent_transaction_count

In [36]:
import numpy as np
import pandas as pd

In [37]:
df = pd.read_csv(r"C:\Users\shali\OneDrive\Desktop\FinanceX\data\FinanceX_transaction.csv")

In [38]:
df.head()

,customer_id,transaction_date,account_type,transaction_amount,transaction_direction,account_balance,merchant_category,emi_amount,transaction_status,channel,transaction_hour
0,CUST015796,2019-01-01,Current,1820.17,Debit,609365.31,Food & Dining,0.00,Success,Branch,15
1,CUST000861,2019-01-01,Current,392.67,Credit,14451.14,Education,0.00,Success,Mobile_App,3
2,CUST076821,2019-01-01,Fixed Deposit,1255.97,Debit,47621.87,Utilities,1433.91,Success,Mobile_App,18
3,CUST054887,2019-01-01,Savings,2580.68,Debit,34467.85,Travel,6280.35,Success,API,8
4,CUST006266,2019-01-01,Fixed Deposit,2573.80,Debit,26617.40,Utilities,0.00,Success,API,14


#### Anomaly Dataset 

In [39]:
df["_original_index"] = range(len(df))
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

In [40]:
# basic feature maping
anomaly_df = pd.DataFrame()
anomaly_df["customer_id"] = df["customer_id"]
anomaly_df["transaction_id"] = df["_original_index"]
anomaly_df["amount"] = df["transaction_amount"]
anomaly_df["category"] = df["merchant_category"]
anomaly_df["balance"] = df["account_balance"]
anomaly_df["transaction_hour"] = df["transaction_hour"]

In [41]:
# category maping
category_mapping = {
    "Food & Dining": "Food",
    "Travel": "Travel",
    "Entertainment": "Entertainment",
    "Retail": "Shopping",
    "E-Commerce": "Shopping"
}

anomaly_df["category"] = (
    df["merchant_category"]
    .map(category_mapping)
    .fillna("Others")
)

In [42]:
# is_weekend feature 
anomaly_df["day_of_week"] = df["transaction_date"].dt.dayofweek

anomaly_df["is_weekend"] = (
    anomaly_df["day_of_week"].isin([5, 6]).astype(int)
)

In [43]:
# User average amount
anomaly_df["user_average_amount"] = (
    df.groupby("customer_id")["transaction_amount"]
      .transform("mean")
)

In [44]:
# Category average amount
anomaly_df["category_average_amount"] = (
    anomaly_df.groupby("category")["amount"]
    .transform("mean")
)

In [45]:
# Amount vs User Average
anomaly_df["amount_vs_user_avg"] = (
    anomaly_df["amount"] /
    anomaly_df["user_average_amount"]
)

In [46]:
# Amount vs Category Average
anomaly_df["amount_vs_category_avg"] = (
    anomaly_df["amount"] /
    anomaly_df["category_average_amount"]
)

In [47]:
# transaction frequency
anomaly_df["transaction_frequency"] = (
    df.groupby("customer_id")["customer_id"]
      .transform("count")
)

In [48]:
# Recent transaction count
temp = df[
    ["_original_index", "customer_id", "transaction_date"]
].copy()

temp = temp.sort_values(
    ["customer_id", "transaction_date"]
)

temp["recent_transaction_count"] = (
    temp.groupby("customer_id")
        .rolling(
            "24h",
            on="transaction_date"
        )["transaction_date"]
        .count()
        .reset_index(level=0, drop=True)
        .values
)

In [49]:
# Restore original order
temp = temp.sort_values("_original_index")

anomaly_df["recent_transaction_count"] = (
    temp["recent_transaction_count"].values
)

In [50]:
anomaly_df = anomaly_df[
    [
        "customer_id",
        "transaction_id",
        "amount",
        "category",
        "user_average_amount",
        "category_average_amount",
        "transaction_frequency",
        "balance",
        "day_of_week",
        "transaction_hour",
        "is_weekend",
        "amount_vs_user_avg",
        "amount_vs_category_avg",
        "recent_transaction_count"
    ]
]

In [51]:
anomaly_df.to_csv(r"C:\Users\shali\OneDrive\Desktop\FinanceX\data\anomaly_dataset.csv", index=False)

In [52]:
anomaly_df.head()

,customer_id,transaction_id,amount,category,user_average_amount,category_average_amount,transaction_frequency,balance,day_of_week,transaction_hour,is_weekend,amount_vs_user_avg,amount_vs_category_avg,recent_transaction_count
0,CUST015796,0,1820.17,Food,34347.187000,30129.894536,10,609365.31,1,15,0,0.052993,0.060411,1.0
1,CUST000861,1,392.67,Others,7653.023750,29932.503880,8,14451.14,1,3,0,0.051309,0.013119,1.0
2,CUST076821,2,1255.97,Others,984.958000,29932.503880,5,47621.87,1,18,0,1.275151,0.041960,1.0
3,CUST054887,3,2580.68,Travel,12239.232857,29608.732225,7,34467.85,1,8,0,0.210853,0.087159,1.0
4,CUST006266,4,2573.80,Others,64137.632000,29932.503880,10,26617.40,1,14,0,0.040129,0.085987,1.0
